# Week 1 Exercise - Technical Question Answerer

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [25]:
# imports
import os
from IPython.display import Markdown, display, update_display
from dotenv import load_dotenv
from openai import OpenAI



In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

OLLAMA_BASE_URL = "http://localhost:11434/v1"


In [4]:
# set up environment

load_dotenv(override=True)
openai = OpenAI()

api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [ ]:
# here is the question; type over this to ask something new

system_prompt = "You are a helpful Software Engineer focused on providing clear and concise explanations of Python Code." \
"You will be given a snippet of code and you will explain what it does overall as well as break down key parts of the code. " \
"You will also add an analysis of time complexity and space complexity of the code." \
"Respond in Markdown format with clear headings for each section of your response. "

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [27]:
# Get gpt-4o-mini to answer, with streaming


def openai_response():
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": question}],
        stream=True
    )
    response=""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)



In [28]:
openai_response()

This Python code snippet utilizes a combination of a generator expression and a set comprehension to yield authors of a collection of books, `books`, under certain conditions. Let's break it down step by step.

### Overall Explanation
The code effectively creates a generator that yields the unique authors from a list of book dictionaries. It filters out any entries that do not have an author and ensures that only distinct authors are yielded.

### Key Parts of the Code

1. **Generator Expression:** 
   - `yield from ...` is used to create a generator. This statement allows you to yield all values from another iterable, in this case, a set comprehension.

2. **Set Comprehension:** 
   - `{book.get("author") for book in books if book.get("author")}`:
     - This part constructs a set of authors. The `for` loop iterates over each `book` in the collection `books`.
     - `book.get("author")` attempts to retrieve the value associated with the key `'author'`. 
     - The `if book.get("author")` condition checks whether the author exists (i.e., it is not `None` or an empty string).
     - Using a set comprehension automatically handles duplicates, thus ensuring that only unique authors are subsequently yielded.

### Example:
If `books` is defined as follows:
```python
books = [
    {"title": "Book A", "author": "Author One"},
    {"title": "Book B", "author": "Author Two"},
    {"title": "Book C", "author": None},
    {"title": "Book D", "author": "Author One"},
]
```
The code will generate:
- `Author One`
- `Author Two`

### Time Complexity Analysis
- **Set Comprehension**: The overall time complexity for iterating through a list of `n` books while performing a `get` operation on each book is O(n). The set creation ensures that this process also has an overhead of O(1) for each insertion, keeping the whole process O(n) in terms of unique insertions.
  
Thus, the time complexity of the code snippet is **O(n)**, where `n` is the number of books in the `books` list.

### Space Complexity Analysis
- **Space for Unique Authors**: The space complexity is determined by the number of unique authors. In the worst case, where all books have different authors, the space complexity can also be considered O(n) since there could be up to `n` unique author names being stored in the set.

Overall, the space complexity of this snippet can be thought of as **O(u)**, where `u` is the number of unique authors, but in terms of the total number of book entries, it would still be close to O(n) in worst-case scenarios.

In summary, this code efficiently provides a generator of unique authors from a list of book dictionaries, with a reasonable expectation for both time and space efficiencies.

In [29]:
# Get Llama 3.2 to answer

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


def ollama_response():
    response = ollama.chat.completions.create(
        model=MODEL_LLAMA, 
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": question}])
    
    result = response.choices[0].message.content
    display(Markdown(result))

In [30]:
ollama_response()

**Overall Explanation**

This line of code is using a combinatorial generator expression to extract authors' names from a list of books, filtered by the presence of the "author" key in each book's dictionary.

**Breaking Down the Code**

Here's how it works:

1. `{}` creates an empty dictionary that acts as a container for the next value yielded.
2. `yield from` is used to delegate part of the control flow of the yielding function (in this case, a generator expression) to other sub-generators (or other yielding functions).
3. `[book.get("author") for book in books if book.get("author")]` defines a list comprehension that:
	* Iterates over each `book` in the `books` iterable.
	* For each `book`, it tries to access the value associated with the key `"author"` using `.get("author")`.
	* If the key exists (i.e., `book.get("author") != None`), its value is included in the sequence. Otherwise, `None` is included.
4. The expression inside `{}` acts as a "funnel" that receives and yields values from the list comprehension.

**Example Use Case**

```python
books = [
    {"title": "Book A", "author": "John Doe"},
    {"title": "Book B" },
    {"title": "Book C", "author": "Jane Smith"}
]

authors = yield from {book.get("author") for book in books if book.get("author")}
# Output: ['John Doe', 'Jane Smith']

for author in authors:
    print(author)
```

**Time Complexity Analysis**

This expression does the following steps:

* Iterates over each book in `books`. If iteration were un-nested, as in linear loops, this would be O(m), but because of how dict get works in Python, it will iterate until a failure to find a "key" is found. This means, if "author" never exists for any element of books, dictionary `{"book": ...}` gets populated with None values instead and then iterating yields one None value at every step where book object would exist.
* Checks the existence of each key. In Python, `.get(key)` operation performs a constant-time lookup ( O(1)).
* Yields values.

Given the fact that iteration does not depend on dictionary keys size but will never exceed it, if books has "N" amount of elements - This operation will always take at most O(m*N) amount of time where 'm' is number of times dict `.get()` function iterates. Because dict get() only hits once whether for each book object value present that would be used to determine whether or not this should exist in final sequence.

**Space Complexity Analysis**

This expression:

* Creates a dictionary `{}` and assigns it as the yielded value, which takes constant space O(1)
* Stores values in `books` list. The storage size depends on how many elements are present in data structure
* Uses additional space for storing variables (`author`, etc.)
 

Overall space complexity: O(N*M)